# Frontend de produccion local

**Objetivo:** preparar una interfaz de uso local para moderar textos o transcripciones completas.

El cuaderno tiene dos salidas:
1. Una funcion Python de inferencia con el modelo entrenado en el cuaderno 04.
2. Un HTML estatico para demo local usando el modelo ligero `modelo_ligero_palabras.json`.

In [ ]:
!pip3 install -q pandas scikit-learn joblib

In [ ]:
from pathlib import Path
import json
import re

import joblib
import pandas as pd

ROOT = Path('..').resolve()
MODELS_DIR = ROOT / 'modelos'
REPORTS_DIR = ROOT / 'resultados'
FRONTEND_DIR = ROOT / 'Cuadernos' / 'frontend'

for path in [REPORTS_DIR, FRONTEND_DIR]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / 'moderador_tfidf_logreg.joblib'
LEXICON_PATH = MODELS_DIR / 'modelo_ligero_palabras.json'
print('Modelo Python:', MODEL_PATH)
print('Modelo ligero:', LEXICON_PATH)

## 1. Inferencia local en Python

Esta seccion usa el modelo entrenado. No requiere servidor ni API.

In [ ]:
def crear_chunks_texto(texto, max_palabras=120):
    palabras = re.findall(r'\S+', texto or '')
    chunks = []
    for i in range(0, len(palabras), max_palabras):
        parte = ' '.join(palabras[i:i + max_palabras])
        if parte.strip():
            chunks.append({'chunk_id': f'chunk_{len(chunks):04d}', 'text': parte})
    return chunks


def predecir_texto(texto, model_path=MODEL_PATH):
    if not model_path.exists():
        raise FileNotFoundError('Entrenar primero el modelo en el cuaderno 04.')
    pack = joblib.load(model_path)
    modelo = pack['model']
    mlb = pack['mlb']
    chunks = crear_chunks_texto(texto)
    pred = modelo.predict([row['text'] for row in chunks])
    etiquetas = mlb.inverse_transform(pred)
    rows = []
    for row, labs in zip(chunks, etiquetas):
        rows.append({**row, 'labels': list(labs) if labs else ['sin_alerta']})
    return pd.DataFrame(rows)


texto_demo = 'Texto de prueba para validar el flujo de produccion local.'
if MODEL_PATH.exists():
    pred_df = predecir_texto(texto_demo)
    pred_df.to_csv(REPORTS_DIR / 'predicciones_demo.csv', index=False)
    display(pred_df)
else:
    print('Modelo no encontrado. Ejecutar cuaderno 04 despues del etiquetado.')

## 2. Frontend HTML externo de produccion

El frontend se mantiene como archivo externo para evitar HTML embebido en el notebook. La demo carga manualmente `modelos/modelo_ligero_palabras.json`, recibe texto, crea chunks y muestra categorias sugeridas.


In [ ]:
from pathlib import Path

html_path = FRONTEND_DIR / 'produccion_moderador.html'
if not html_path.exists():
    raise FileNotFoundError(f'No existe el frontend: {html_path}')

print('Frontend de produccion disponible en:', html_path)


## 3. Checklist de despliegue local

- `modelos/moderador_tfidf_logreg.joblib` existe despues del entrenamiento.
- `modelos/modelo_ligero_palabras.json` existe para la demo HTML.
- El HTML se abre localmente y no realiza llamadas externas.
- Los resultados deben tratarse como sugerencias y pasar por revision humana.